In [1]:
# ---------------- Imports ----------------
import json
import os

import numpy as np
import pandas as pd
import yaml

from sentence_transformers import CrossEncoder
from result_groups import model_set_comparison, k_comparison, model_set_comparison_neutralprompt

import multiprocessing as mp
from tqdm.auto import tqdm

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")



In [2]:
# ---------------- Inputs ----------------

#FILES = k_comparison.FILES
#FILES = model_set_comparison.FILES
FILES = model_set_comparison_neutralprompt.FILES


MODEL_NAME = "Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt"
HISTORY_FILES = FILES[MODEL_NAME]


SIMILARITY_MODEL = "cross-encoder/stsb-roberta-large"
SIMILARITY_THRESHOLD = 0.7



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
MODELS_PATH = config["paths"]["models"]

INPUT_FOLDER = os.path.join(PROJ_STORE, "experiments", "ociv-trajectory")


#OUTPUT_FOLDER = os.path.join(PROJ_STORE, "evaluation", "redundancy-test")
OUTPUT_FOLDER = os.path.join(PROJ_STORE, "evaluation", "redundancy-test", MODEL_NAME)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
#OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "candidate-score-changes.csv")

SIMILARITY_MODEL_PATH = os.path.join(MODELS_PATH, SIMILARITY_MODEL)





In [4]:
# ---------------- Load data ----------------
def load_trajectory_file(path):
    full_path = os.path.join(INPUT_FOLDER, path)

    with open(full_path, "r", encoding="utf-8") as f:
        return json.load(f)


# ---------------- Load records ----------------
datasets = [load_trajectory_file(path) for path in HISTORY_FILES]

num_records = sum(
    len(trajectory["steps"])
    for data in datasets
    for trajectory in data["trajectories"]
)

print(f"{MODEL_NAME}")
print(f"Loaded {num_records:,} trajectory-step records")



Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt
Loaded 30,000 trajectory-step records


In [5]:
# ---------------- Build candidate-history pairs ----------------
pair_rows = []
pairs = []

for data in datasets:
    positive_label = data["params"]["positive_label"]
    negative_label = data["params"]["negative_label"]
    seed = data["params"]["seed"]

    for trajectory in data["trajectories"]:
        trajectory_id = trajectory["trajectory_id"]

        for record in trajectory["steps"]:
            for candidate in record["candidates"]:
                history = candidate["history"]

                if not history:
                    continue

                for history_entry in history:
                    pairs.append(
                        (
                            history_entry["argument"],
                            candidate["argument"],
                        )
                    )

                    pair_rows.append({
                        "seed": seed,
                        "trajectory_id": trajectory_id,
                        "step": record["step"],
                        "positive_label": positive_label,
                        "negative_label": negative_label,
                        "argument_id": candidate["argument_id"],
                        "candidate": candidate["argument"],
                        "candidate_topic": candidate["topic"],
                        "history_argument_id": history_entry["argument_id"],
                        "history_argument": history_entry["argument"],
                        "history_topic": history_entry["topic"],
                        "history_confirmed": history_entry["confirmed"],
                        "positive_prob_with_history": candidate["positive_prob_with_history"],
                        "positive_prob_without_history": candidate["positive_prob_without_history"],
                        "selected_candidate": candidate["selected_candidate"],
                    })

print(f"Scoring {len(pairs):,} candidate-history pairs")



Scoring 312,556 candidate-history pairs


In [6]:
def score_chunk(args):
    gpu_id, chunk, progress_queue = args
    model = CrossEncoder(SIMILARITY_MODEL_PATH, device=f"cuda:{gpu_id}")

    batch_size = 1024
    scores = []

    for start in range(0, len(chunk), batch_size):
        batch = chunk[start:start + batch_size]
        batch_scores = model.predict(
            batch,
            batch_size=batch_size,
            show_progress_bar=False,
        )
        scores.extend(batch_scores)
        progress_queue.put(len(batch))

    return np.asarray(scores)



num_gpus = 3
pair_chunks = np.array_split(np.array(pairs, dtype=object), num_gpus)
pair_chunks = [chunk.tolist() for chunk in pair_chunks]

ctx = mp.get_context("fork")
manager = ctx.Manager()
progress_queue = manager.Queue()

with ctx.Pool(num_gpus) as pool:
    jobs = [
        pool.apply_async(score_chunk, ((gpu_id, chunk, progress_queue),))
        for gpu_id, chunk in enumerate(pair_chunks)
    ]

    with tqdm(total=len(pairs), unit="pairs") as progress:
        completed = 0

        while completed < len(pairs):
            n = progress_queue.get()
            completed += n
            progress.update(n)

    chunk_scores = [job.get() for job in jobs]

scores = np.concatenate(chunk_scores)



  0%|          | 0/312556 [00:00<?, ?pairs/s]

In [7]:
# ---------------- Build scored dataframe ----------------

df = pd.DataFrame(pair_rows)
df["similarity_score"] = scores
df["similarity_gt_threshold"] = df["similarity_score"] > SIMILARITY_THRESHOLD
df["positive_prob_difference"] = df["positive_prob_without_history"] - df["positive_prob_with_history"]

# check
display(df.head(2))



,seed,trajectory_id,step,positive_label,negative_label,argument_id,candidate,candidate_topic,history_argument_id,history_argument,history_topic,history_confirmed,positive_prob_with_history,positive_prob_without_history,selected_candidate,similarity_score,similarity_gt_threshold,positive_prob_difference
0,42,0,1,Yes,No,18312,we should legalize polygamy as it is persons right to choose their number of spouses and more than one spouse can lead to an increased sense of family and community.,We should legalize polygamy,11383,polygamy puts the needs and wants of the man more important than that of the womens,We should legalize polygamy,True,0.956,0.989,False,0.335,False,0.033
1,42,0,1,Yes,No,20624,women in a polygamous marriage tend to become submissive for fear of being replaced.,We should legalize polygamy,11383,polygamy puts the needs and wants of the man more important than that of the womens,We should legalize polygamy,True,0.984,0.994,False,0.505,False,0.010


In [8]:
scored_pairs_path = os.path.join(OUTPUT_FOLDER, f"{MODEL_NAME}-scored-pairs.csv")
df.to_csv(scored_pairs_path, index=False)

print(f"Saved scored pairs to {scored_pairs_path}")


Saved scored pairs to /data/ociv-agents/evaluation/redundancy-test/Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt/Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt-scored-pairs.csv


In [9]:
# To import rather than recompute

#scored_pairs_path = os.path.join(OUTPUT_FOLDER, f"{MODEL_NAME}-scored-pairs.csv")
#df = pd.read_csv(scored_pairs_path)



In [ ]:
# ---------------- Main summary ----------------
summary = (
    df
    .groupby(
        [
            "similarity_gt_threshold",
            "history_confirmed",
        ]
    )
    .agg(
        n=("positive_prob_difference", "size"),
        mean_similarity=("similarity_score", "mean"),
        mean_without_history=("positive_prob_without_history", "mean"),
        mean_with_history=("positive_prob_with_history", "mean"),
        mean_prob_difference=("positive_prob_difference", "mean"),
    )
    .reset_index()
)

summary["similarity_group"] = summary["similarity_gt_threshold"].map({
    False: f"<= {SIMILARITY_THRESHOLD}",
    True: f"> {SIMILARITY_THRESHOLD}",
})

summary["history_outcome"] = summary["history_confirmed"].map({
    True: "confirmed",
    False: "not confirmed",
})

summary = summary[
    [
        "similarity_group",
        "history_outcome",
        "n",
        "mean_similarity",
        "mean_with_history",
        "mean_without_history",
        "mean_prob_difference",
    ]
]

display(summary)

summary_output_path = os.path.join(OUTPUT_FOLDER, f"{MODEL_NAME}-summary.csv")
summary.to_csv(summary_output_path, index=False)



,similarity_group,history_outcome,n,mean_similarity,mean_with_history,mean_without_history,mean_prob_difference
0,<= 0.7,not confirmed,151111,0.309,0.889,0.979,0.089
1,<= 0.7,confirmed,152404,0.308,0.893,0.978,0.085
2,> 0.7,not confirmed,4516,0.769,0.950,0.986,0.035
3,> 0.7,confirmed,4525,0.769,0.962,0.987,0.024


In [ ]:
# ---------------- Inspect high-similarity pairs ----------------
display_columns = [
    "trajectory_id",
    "step",
    "candidate",
    "history_argument",
    "similarity_score",
    "positive_prob_without_history",
    "positive_prob_with_history",
    "positive_prob_difference",
    "selected_candidate",
]

high_similarity_df = (
    df[df["similarity_score"] > SIMILARITY_THRESHOLD][display_columns]
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

display(high_similarity_df.head(5))



,trajectory_id,step,candidate,history_argument,similarity_score,positive_prob_without_history,positive_prob_with_history,positive_prob_difference,selected_candidate
0,67,24,we shouldn't ban the use of child actors because it allows the kids to pursue a career early in their life,we shouldn't ban the use of child actors because it is a way for children to find a career early in their life,0.975,0.990,0.955,0.035,False
1,28,63,we should not abandon marriage because some people still believe in the wedding vows,we should not abandon marriage because there are still a lot of people that believe in marriage vows,0.973,0.992,0.903,0.088,False
2,28,61,not everyone identifies as a male or female so our pronouns should reflect that,not everyone identifies as a man or a woman and our pronouns should reflect that,0.973,0.992,0.958,0.033,False
3,40,27,student loans should be subsidized because it is unfair to saddle graduates with a heavy debt burden when they are starting out on their chosen careers,student loans should be subsidized because it is unfair to saddle students with crippling amount of debts as they start out on their career paths,0.972,0.998,0.996,0.002,False
4,1,83,people should be allowed to work as long as they like,people should be able to work as long as they want to.,0.970,0.995,0.985,0.011,False


In [12]:
# ---------------- Build candidate-level similarity data ----------------
candidate_rows = []

for data in datasets:
    seed = data["params"]["seed"]

    for trajectory in data["trajectories"]:
        trajectory_id = trajectory["trajectory_id"]

        for record in trajectory["steps"]:
            candidates = record["candidates"]
            without_scores = np.array([candidate["positive_prob_without_history"] for candidate in candidates])
            selected_without_position = int(np.argmax(without_scores))

            for candidate in candidates:
                candidate_rows.append({
                    "seed": seed,
                    "trajectory_id": trajectory_id,
                    "step": record["step"],
                    "argument_id": candidate["argument_id"],
                    "candidate": candidate["argument"],
                    "topic": candidate["topic"],
                    "positive_prob_with_history": candidate["positive_prob_with_history"],
                    "positive_prob_without_history": candidate["positive_prob_without_history"],
                    "positive_prob_difference": candidate["positive_prob_without_history"] - candidate["positive_prob_with_history"],
                    "selected_with_history": candidate["selected_candidate"],
                    "selected_without_history": candidate["candidate_position"] == selected_without_position,
                })

candidate_df = pd.DataFrame(candidate_rows)



In [13]:
# ---------------- Aggregate similarity scores to candidate level ----------------
keys = ["seed", "trajectory_id", "step", "argument_id"]

similarity_summary = df.groupby(keys).agg(max_similarity=("similarity_score", "max")).reset_index()

confirmed_summary = (
    df[df["history_confirmed"]]
    .groupby(keys)["similarity_score"]
    .max()
    .rename("max_confirmed_similarity")
    .reset_index()
)

unconfirmed_summary = (
    df[~df["history_confirmed"]]
    .groupby(keys)["similarity_score"]
    .max()
    .rename("max_unconfirmed_similarity")
    .reset_index()
)

max_indices = df.groupby(keys)["similarity_score"].idxmax()
most_similar = df.loc[max_indices, keys + ["history_confirmed"]].copy()
most_similar = most_similar.rename(columns={"history_confirmed": "most_similar_confirmed"})

candidate_df = candidate_df.merge(similarity_summary, on=keys, how="left")
candidate_df = candidate_df.merge(confirmed_summary, on=keys, how="left")
candidate_df = candidate_df.merge(unconfirmed_summary, on=keys, how="left")
candidate_df = candidate_df.merge(most_similar, on=keys, how="left")


In [14]:
# ---------------- Selection changes ----------------
candidate_df["selection_change"] = np.select(
    [
        candidate_df["selected_without_history"]
        & ~candidate_df["selected_with_history"],
        ~candidate_df["selected_without_history"]
        & candidate_df["selected_with_history"],
    ],
    [
        "deselected_by_history",
        "selected_by_history",
    ],
    default="unchanged",
)



In [15]:
# ---------------- Nearest-history analysis ----------------
high_sim = candidate_df[
    candidate_df["max_similarity"] > SIMILARITY_THRESHOLD
]

selection_summary = (
    high_sim
    .groupby("most_similar_confirmed")
    .agg(
        n=("argument_id", "size"),
        selected_with_history=("selected_with_history", "sum"),
        selected_without_history=("selected_without_history", "sum"),
        selection_rate_with_history=("selected_with_history", "mean"),
        selection_rate_without_history=("selected_without_history", "mean"),
    )
    .reset_index()
)

display(selection_summary)



,most_similar_confirmed,n,selected_with_history,selected_without_history,selection_rate_with_history,selection_rate_without_history
0,False,3901,461,659,0.118,0.169
1,True,3915,579,674,0.148,0.172


In [16]:
selection_changes = (
    high_sim
    .groupby(["most_similar_confirmed", "selection_change"])
    .size()
    .reset_index(name="n")
)

display(selection_changes)




,most_similar_confirmed,selection_change,n
0,False,deselected_by_history,464
1,False,selected_by_history,266
2,False,unchanged,3171
3,True,deselected_by_history,423
4,True,selected_by_history,328
5,True,unchanged,3164


In [17]:
# ---------------- Confirmed/unconfirmed similarity groups ----------------
candidate_df["high_similarity_confirmed"] = (
    candidate_df["max_confirmed_similarity"] > SIMILARITY_THRESHOLD
)

candidate_df["high_similarity_unconfirmed"] = (
    candidate_df["max_unconfirmed_similarity"] > SIMILARITY_THRESHOLD
)

candidate_df["similarity_group"] = np.select(
    [
        candidate_df["high_similarity_confirmed"],
        ~candidate_df["high_similarity_confirmed"]
        & candidate_df["high_similarity_unconfirmed"],
    ],
    [
        "similar to confirmed",
        "similar only to unconfirmed",
    ],
    default="no high similarity",
)

grouped_high_sim = candidate_df[
    candidate_df["similarity_group"] != "no high similarity"
]



In [18]:
selection_summary = (
    grouped_high_sim
    .groupby("similarity_group")
    .agg(
        n=("argument_id", "size"),
        selected_with_history=("selected_with_history", "sum"),
        selected_without_history=("selected_without_history", "sum"),
        selection_rate_with_history=("selected_with_history", "mean"),
        selection_rate_without_history=("selected_without_history", "mean"),
        mean_prob_difference=("positive_prob_difference", "mean"),
    )
    .reset_index()
)

display(selection_summary)

selection_output_path = os.path.join(OUTPUT_FOLDER, f"{MODEL_NAME}-selection-summary.csv")
selection_summary.to_csv(selection_output_path, index=False)

print(f"Saved to {selection_output_path}")


,similarity_group,n,selected_with_history,selected_without_history,selection_rate_with_history,selection_rate_without_history,mean_prob_difference
0,similar only to unconfirmed,3682,427,604,0.116,0.164,0.039
1,similar to confirmed,4134,613,729,0.148,0.176,0.026


Saved to /data/ociv-agents/evaluation/redundancy-test/Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt/Mistral-Small-3.2-24B-Instruct-2506t-neutralprompt-selection-summary.csv
